In [7]:
from pathlib import Path

# import anndata as ad
import dask.array as da
import matplotlib.pyplot as plt
import napari
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm
import os, glob
from skimage import io
import tarrow
import torch
import tifffile

expanded_piyg = ['#1a9641', '#a6d96a', '#978897', '#d1d1ca', '#f1b6da', '#d02c91']


def timed_compute(volume):
    """
    Compute a lazy Dask array frame-by-frame with progress reporting.

    This function iterates over the leading axis of a Dask array (e.g. time),
    calls `.compute()` on each slice, and stacks the results into a single
    NumPy array. A tqdm progress bar is displayed to indicate progress.

    Parameters
    ----------
    volume : dask.array.Array
        A Dask array with at least one dimension (e.g. shape (T, ...)).
        The function will iterate over the first axis (axis=0).

    Returns
    -------
    numpy.ndarray
        A NumPy array with the same shape as `volume`, but fully realized
        in memory. The dtype is preserved from the Dask array.

    Notes
    -----
    - Each frame is computed independently, which can be helpful for
      monitoring performance and memory use on large arrays.
    - The returned array may be very large if `volume` is large.
      Ensure sufficient memory is available.
    """
    return np.stack([frame.compute() for frame in tqdm(volume)], axis=0)


In [3]:
root_dir = Path('/mnt/OPERA3/Nathan/Z_stack_tests/zarr')
# root_dir = Path('/Volumes/OPERA3/Nathan/Z_stack_tests/zarr')
rc_stem = "(3,3)"
store = root_dir / f"{rc_stem}.zarr"

# --- load images and segmentation ---
images = da.from_zarr(str(store / "images" / "0"))   # (T,C,Z,Y,X)
masks  = da.from_zarr(str(store / "labels" / "masks"))  # (T,Z,Y,X)

# --- load tracks (CSV preferred, fallback to AnnData) ---
# tracks_csv = root_dir / f"{rc_stem}_tracks.csv"
# if tracks_csv.exists():
#     tracks = pd.read_csv(tracks_csv)
# else:
# adata = ad.read_zarr(str(store / "tables" / "quantified_tracks"))
# tracks = adata.obs.reset_index(drop=True)

print("images:", images.shape)
print("masks:", masks.shape)
# print("tracks:", tracks.shape)


images: (97, 2, 25, 7992, 7992)
masks: (97, 25, 7992, 7992)


## Working with a subset of data

In [20]:
x0, x1, y0, y1 = 4270, 6047, 2150, 4060

In [21]:
images_roi = images[:, :, :, y0:y1, x0:x1]
# masks_roi  = masks[:, :, y0:y1, x0:x1]


In [22]:
images_roi = timed_compute(images_roi)

  0%|          | 0/97 [00:00<?, ?it/s]

In [23]:
%%time
input_image = images_roi[:,1,...].max(axis=1)

CPU times: user 2.31 s, sys: 64.1 ms, total: 2.37 s
Wall time: 2.37 s


In [24]:
input_image.dtype, input_image.shape

(dtype('uint16'), (97, 1910, 1777))

In [17]:
import cv2

In [25]:
scaled_image = cv2.convertScaleAbs(input_image, alpha=(255.0/np.max(input_image)))


# Train model

In [38]:
input_image.shape, input_image.dtype

((97, 1910, 1777), dtype('uint8'))

In [39]:
os.mkdir('/mnt/OPERA3/Nathan/Z_stack_tests/TAP')

In [41]:
io.imsave('/mnt/OPERA3/Nathan/Z_stack_tests/TAP/z_depth_tests_timelapse_gfp_max_proj.tif', input_image)

/tmp/ipykernel_147943/7823587.py:1: UserWarning: /mnt/OPERA3/Nathan/Z_stack_tests/TAP/z_depth_tests_timelapse_gfp_max_proj.tif is a low contrast image
  io.imsave('/mnt/OPERA3/Nathan/Z_stack_tests/TAP/z_depth_tests_timelapse_gfp_max_proj.tif', input_image)


# Apply model

In [8]:
model = tarrow.models.TimeArrowNet.from_folder(model_folder='/home/dayn/analysis/models/tarrow/ND0000/macro_backbone_unet/')

In [9]:
model

TimeArrowNet(
  (backbone): Unet2d(
    (l_conv): ModuleList(
      (0): ConvBlock(
        (conv_block): Sequential(
          (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), padding_mode=replicate)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): LeakyReLU(negative_slope=0.01, inplace=True)
          (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), padding_mode=replicate)
          (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (5): LeakyReLU(negative_slope=0.01, inplace=True)
        )
      )
      (1): ConvBlock(
        (conv_block): Sequential(
          (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), padding_mode=replicate)
          (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): LeakyReLU(negative_slope=0.01, inplace=True)
          (3): Conv2d(64, 64,

In [26]:
# viewer = napari.Viewer(title='TAP tests')
viewer.add_image(scaled_image)

<Image layer 'scaled_image [1]' at 0x79750cf18eb0>

In [29]:
 input_image.shape

(97, 1910, 1777)

In [30]:
# Dummy data: Batch, Time, Channel, X, Y
x = torch.from_numpy(input_image).unsqueeze(0).unsqueeze(2)

In [35]:
# x: (Batch, Time, Channel, X, Y) — float32 on same device as model
device = next(model.parameters()).device
device

device(type='cpu')

In [32]:


x = torch.from_numpy(input_image)                # (T, X, Y), likely uint8
if x.dtype == torch.uint8:
    x = x.float().div_(255)                      # to float in [0,1]
else:
    x = x.float()

x = x.unsqueeze(0).unsqueeze(2)                  # -> (1, T, 1, X, Y)
x = x.to(device, non_blocking=True)

model.eval()                                     # avoid BN issues with batch=1
with torch.inference_mode():
    rep = model.embedding(x)
print(tuple(rep.shape))


torch.Size([1, 97, 1, 1910, 1777])

In [33]:

# Dense dummy representations
rep = model.embedding(x)
print(f"Dense representations for image `x` with shape {tuple(rep.shape)}")

RuntimeError: Input type (unsigned char) and bias type (float) should be the same

In [ ]:
input_tensor.shape

In [ ]:
viewer.add_image(input_image)

In [ ]:
%%time
rep = model.embedding(input_tensor)
print(f"Dense representations for image volume with shape {tuple(rep.shape)}")

In [ ]:
np_rep = rep[0,0,...].detach().numpy()

In [ ]:
np_rep.shape

In [ ]:
np_rep_sum = np.sum(np_rep, axis = 0)

In [ ]:
viewer.add_image(input_image, blending = 'additive')
viewer.add_image(np_rep_sum, blending = 'additive')

In [ ]:
viewer.add_image(np_rep)

In [43]:
viewer.add_image(np_rep, channel_axis=0, blending='additive')

[<Image layer 'Image' at 0x7fdf7256f850>,
 <Image layer 'Image [1]' at 0x7fdf72597850>,
 <Image layer 'Image [2]' at 0x7fdf722d20d0>,
 <Image layer 'Image [3]' at 0x7fdf7c580430>,
 <Image layer 'Image [4]' at 0x7fdf684d25b0>,
 <Image layer 'Image [5]' at 0x7fdf66239880>,
 <Image layer 'Image [6]' at 0x7fdf79c0fb50>,
 <Image layer 'Image [7]' at 0x7fdf74e9e610>,
 <Image layer 'Image [8]' at 0x7fdf7c8e5880>,
 <Image layer 'Image [9]' at 0x7fdf88752df0>,
 <Image layer 'Image [10]' at 0x7fdf8877fe20>,
 <Image layer 'Image [11]' at 0x7fdf88730d30>,
 <Image layer 'Image [12]' at 0x7fdf886dcd60>,
 <Image layer 'Image [13]' at 0x7fdf8870cc70>,
 <Image layer 'Image [14]' at 0x7fdf886b8ca0>,
 <Image layer 'Image [15]' at 0x7fdf88669bb0>,
 <Image layer 'Image [16]' at 0x7fdf88614be0>,
 <Image layer 'Image [17]' at 0x7fdf88646af0>,
 <Image layer 'Image [18]' at 0x7fdf885f0b20>,
 <Image layer 'Image [19]' at 0x7fdf63554a30>,
 <Image layer 'Image [20]' at 0x7fdf63500a60>,
 <Image layer 'Image [21]' 

# Run over all frames?

In [22]:
from tqdm.auto import tqdm

In [24]:
images = da.sum(images, axis = 2)[:,0]

In [13]:
images.shape

(1194, 1200, 1600)

In [ ]:
import numpy as np
import torch
from tqdm.auto import tqdm

# Define the step size (e.g., every tenth frame)
step_size = 1
num_frames = len(images)
rep_sums = []
channel = 0

for t in tqdm(range(0, num_frames, step_size), total=num_frames // step_size):
    input_image = images[t].compute().astype(np.int16)
    input_tensor = torch.Tensor(input_image[None, None, None])

    with torch.no_grad():
        rep = model.embedding(input_tensor)
    
    np_rep = rep[0, 0, ...].cpu().numpy()
    np_rep_sum = np.sum(np_rep, axis=0)
    rep_sums.append(np_rep_sum)

rep_sums = np.stack(rep_sums, axis=0)


  0%|          | 0/376 [00:00<?, ?it/s]

In [16]:
viewer = napari.Viewer(title = 'tarrow over every frame')
viewer.add_image(selected_frames_volume, blending = 'additive')
viewer.add_image(rep_sums, blending = 'additive')

<Image layer 'rep_sums' at 0x7f5669c91670>

In [17]:
np.save('/home/dayn/analysis/models/tarrow/reduced_batchsize/macro_backbone_unet/visuals/all_frames.npy', selected_frames)
np.save('/home/dayn/analysis/models/tarrow/reduced_batchsize/macro_backbone_unet/visuals/all_frames_rep_sums.npy', rep_sums)

In [ ]:
rep_sums = np.load('/home/dayn/analysis/models/tarrow/reduced_batchsize/macro_backbone_unet/visuals/all_frames_rep_sums.npy')

In [ ]:
rep_sums.shape

In [ ]:
viewer_2 = napari.Viewer(title='previous TAP work')
viewer_2.add_image(rep_sums)

# MDCK example

In [7]:
mdck = images = io.imread('/home/dayn/analysis/tarrow/scripts/data/mdck.tif')

In [6]:
model = tarrow.models.TimeArrowNet.from_folder(model_folder='/home/dayn/analysis/tarrow/scripts/data/runs/10-20-16-56-38_None_backbone_unet/')

INFO:root:Loading model from /home/dayn/analysis/tarrow/scripts/data/runs/10-20-16-56-38_None_backbone_unet


In [8]:
import numpy as np
import torch
from tqdm.auto import tqdm

# Define the step size (e.g., every tenth frame)
step_size = 1
num_frames = len(images)
rep_sums = []
channel = 0

for t in tqdm(range(0, num_frames, step_size), total=num_frames // step_size):
    input_image = images[t]#.compute().astype(np.int16)
    input_tensor = torch.Tensor(input_image[None, None, None])

    with torch.no_grad():
        rep = model.embedding(input_tensor)
    
    np_rep = rep[0, 0, ...].cpu().numpy()
    np_rep_sum = np.sum(np_rep, axis=0)
    rep_sums.append(np_rep_sum)

rep_sums = np.stack(rep_sums, axis=0)


  0%|          | 0/1194 [00:00<?, ?it/s]

In [10]:
np.save('/home/dayn/analysis/tarrow/scripts/data/mdck_reps.npy', rep_sums)

In [11]:
viewer = napari.Viewer()

viewer.add_image(mdck)
viewer.add_image(rep_sums)



<Image layer 'rep_sums' at 0x7f0b9cae6700>

Rendering frames...


100%|██████████████████████████████████████████████████████| 1194/1194 [01:15<00:00, 15.73it/s]
